# German NER — julian/roberta Benchmark

Model: `julian-schelb/roberta-ner-multilingual`  
Datasets: WikiANN German (PAN-X), GermEval 2014

## Setup

**Google Colab:** run the install cell below.  
**Local:** `pip install git+https://github.com/ay94/multilingual-ner.git`

In [ ]:
%%capture
!pip install git+https://github.com/ay94/multilingual-ner.git transformers datasets seqeval sentencepiece

In [ ]:
from multilingual_ner.evaluation import ReadNERData, ModelEvaluation, align_dataset, check_labels

## Load datasets

In [ ]:
xtreme_label_map = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}

reader = ReadNERData()
xtreme_words, xtreme_labels = reader.read_dataset('wikiann', xtreme_label_map, lang='de')
print(f'Loaded {len(xtreme_words)} sentences')
print('Labels:', check_labels(xtreme_labels))

In [ ]:
germeval_label_map = {
    'O': 0,
    'B-LOC': 1, 'I-LOC': 2, 'B-LOCderiv': 3, 'I-LOCderiv': 4, 'B-LOCpart': 5, 'I-LOCpart': 6,
    'B-ORG': 7, 'I-ORG': 8, 'B-ORGderiv': 9, 'I-ORGderiv': 10, 'B-ORGpart': 11, 'I-ORGpart': 12,
    'B-OTH': 13, 'I-OTH': 14, 'B-OTHderiv': 15, 'I-OTHderiv': 16, 'B-OTHpart': 17, 'I-OTHpart': 18,
    'B-PER': 19, 'I-PER': 20, 'B-PERderiv': 21, 'I-PERderiv': 22, 'B-PERpart': 23, 'I-PERpart': 24
}

germeval_label_alignment = {
    'B-LOCderiv': 'B-LOC', 'I-LOCderiv': 'I-LOC',
    'B-LOCpart':  'B-LOC', 'I-LOCpart':  'I-LOC',
    'B-ORGderiv': 'B-ORG', 'I-ORGderiv': 'I-ORG',
    'B-ORGpart':  'B-ORG', 'I-ORGpart':  'I-ORG',
    'B-PERderiv': 'B-ORG', 'I-PERderiv': 'I-ORG',
    'B-PERpart':  'B-ORG', 'I-PERpart':  'I-ORG',
    'B-OTHderiv': 'O',     'I-OTHderiv': 'O',
    'B-OTHpart':  'O',     'I-OTHpart':  'O',
    'B-OTH':      'O',     'I-OTH':      'O',
}

germeval_reader = ReadNERData()
germeval_words, germeval_labels = germeval_reader.read_dataset('germeval_14', germeval_label_map)
germeval_labels = align_dataset(germeval_labels, germeval_label_alignment)
print(f'Loaded {len(germeval_words)} sentences')
print('Labels after alignment:', check_labels(germeval_labels))

## Load model & define alignment

In [ ]:
model_alignment = {
    'B-PER': 'B-PER', 'I-PER': 'I-PER',
    'B-LOC': 'B-LOC', 'I-LOC': 'I-LOC',
    'B-ORG': 'B-ORG', 'I-ORG': 'I-ORG',
    'B-MISC': 'O',    'I-MISC': 'O',
    'O': 'O',
}

model = ModelEvaluation('julian-schelb/roberta-ner-multilingual', model_alignment)
print(model.model.config.id2label)

## Evaluate — WikiANN German

In [ ]:
results_model_xtreme = model.evaluate_model(xtreme_words, xtreme_labels)

print('--- Seqeval (entity-level) ---')
display(results_model_xtreme.get_classification('Seqeval'))

print('--- Sklearn (token-level) ---')
display(results_model_xtreme.get_classification('Sklearn'))

## Evaluate — GermEval 2014

In [ ]:
results_model_germeval = model.evaluate_model(germeval_words, germeval_labels)

print('--- Seqeval (entity-level) ---')
display(results_model_germeval.get_classification('Seqeval'))

print('--- Sklearn (token-level) ---')
display(results_model_germeval.get_classification('Sklearn'))